# 90 · Upload replication data to Harvard Dataverse

Exports the BigQuery tables listed in step 3 to Parquet, adds the local release folder (README, `results/`, `supplementary/`), writes a schema and a checksum manifest, uploads everything into an existing draft dataset and checks every file against the server. Publishing is done in the web interface after the check in step 8 passes.

Requires a draft dataset (persistent identifier), an API token in Colab Secrets as `DV_TOKEN`, and `dataverse_release.zip`.

In [ ]:
# 1 - Configuration
PROJECT_ID  = "your-gcp-project"          # GCP project that holds the dataset
DATASET     = "qbtc_raw"
LOCATION    = "US"
DV_URL      = "https://dataverse.harvard.edu/"
PID         = "doi:10.7910/DVN/XXXXXX"    # persistent identifier of the draft dataset
RELEASE_ZIP = "dataverse_release.zip"
WORK        = "/content/pkg"
CONFIRM_UPLOAD = False                    # set to True after checking the listing in step 6

assert PROJECT_ID != "your-gcp-project" and "XXXXXX" not in PID, "Fill in PROJECT_ID and PID"

In [ ]:
# 2 - Setup
!pip -q install dvuploader
from google.colab import auth, files, userdata
auth.authenticate_user()

import hashlib, os, pathlib, re, zipfile
import pandas as pd, pyarrow as pa, pyarrow.parquet as pq, requests
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DV_TOKEN = userdata.get("DV_TOKEN")
HDR = {"X-Dataverse-key": DV_TOKEN}
root = pathlib.Path(WORK); root.mkdir(parents=True, exist_ok=True)

def draft_files():
    r = requests.get(f"{DV_URL}api/datasets/:persistentId/versions/:draft/files",
                     params={"persistentId": PID}, headers=HDR, timeout=120)
    r.raise_for_status()
    return r.json()["data"]

print("Draft reachable; files already in draft:", len(draft_files()))

In [ ]:
# 3 - Tables to export (all must exist; sizes are checked before anything is downloaded)
TABLES = {
    "inputs": ["events", "attention_spike_audit", "wiki_frozen", "gtrends_frozen", "gdelt_daily",
               "exchange_addresses_raw", "large_movers_labeled", "custodial_keys", "patoshi_blocks"],
    "analysis": ["attention_daily", "event_attention", "attention_spikes", "placebo_days",
                 "placebo_draws", "placebo_draws_strict", "flow_days", "event_flow_coverage",
                 "exchange_keys", "panel_event", "panel_cem_strata", "panel_rs2", "panel_rs2_ext",
                 "events_rs", "flow_days_rs", "placebo_days_rs", "supply_by_type"],
}
MAX_GB = 3.0

meta = []
for tier, names in TABLES.items():
    for t in names:
        tb = client.get_table(f"{PROJECT_ID}.{DATASET}.{t}")
        meta.append(dict(table=t, tier=tier, rows=tb.num_rows, bq_gb=round(tb.num_bytes / 1e9, 3),
                         bq_modified=tb.modified.isoformat()))
meta = pd.DataFrame(meta)
display(meta)
print(f"{len(meta)} tables, {meta.bq_gb.sum():.2f} GB in BigQuery")
assert (meta.bq_gb < MAX_GB).all(), "A table is larger than expected; check before exporting"

In [ ]:
# 4 - Export to Parquet (Zstandard) and check row counts and text
CJK = re.compile(r"[\u3000-\u303f\u4e00-\u9fff\uff00-\uffef]")
cols, checks = [], []
for r in meta.itertuples():
    tbl = client.list_rows(client.get_table(f"{PROJECT_ID}.{DATASET}.{r.table}")).to_arrow(
        create_bqstorage_client=True)
    out = root / "data" / r.tier / f"{r.table}.parquet"
    out.parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(tbl, out, compression="zstd")
    n = pq.ParquetFile(out).metadata.num_rows
    assert n == r.rows, f"{r.table}: {n} rows in Parquet, {r.rows} in BigQuery"
    cjk_cols = [f.name for f in tbl.schema
                if (pa.types.is_string(f.type) or pa.types.is_large_string(f.type))
                and any(CJK.search(v) for v in tbl.column(f.name).drop_null().unique().to_pylist())]
    cols += [dict(table=r.table, column=f.name, type=str(f.type)) for f in tbl.schema]
    checks.append(dict(table=r.table, parquet_rows=n, parquet_mb=round(out.stat().st_size / 1e6, 2),
                       non_english_columns=";".join(cjk_cols)))
    print(f"{r.table:<24}{n:>12,} rows {out.stat().st_size / 1e6:>9.1f} MB  {';'.join(cjk_cols)}")
    del tbl
checks = pd.DataFrame(checks)
bad = checks[checks.non_english_columns != ""]
assert bad.empty, f"Chinese text found in: {bad[['table', 'non_english_columns']].values.tolist()}"

In [ ]:
# 5 - Add the local release folder (README.md, results/, supplementary/)
up = files.upload()
assert RELEASE_ZIP in up, f"Expected {RELEASE_ZIP}, got {list(up)}"
with zipfile.ZipFile(RELEASE_ZIP) as z:
    names = z.namelist()
    assert not any(n.startswith(("data/", "MANIFEST")) for n in names), "Zip must not contain data/ or MANIFEST"
    z.extractall(root)
os.remove(RELEASE_ZIP)
need = ["README.md", "results/est_20_pooled.csv", "results/out40/q3_throughput.csv",
        "supplementary/Data_S1_exchange_addresses.csv", "supplementary/Data_S2_large_movers_labeled.csv"]
missing = [p for p in need if not (root / p).exists()]
assert not missing, missing
print(len(names), "files from the zip")

In [ ]:
# 6 - Schema, manifest and listing
(root / "data" / "schema").mkdir(parents=True, exist_ok=True)
meta.merge(checks, on="table")[["table", "tier", "rows", "parquet_rows", "bq_gb", "parquet_mb", "bq_modified"]] \
    .to_csv(root / "data/schema/tables.csv", index=False)
pd.DataFrame(cols).to_csv(root / "data/schema/columns.csv", index=False)

def digest(p, algo):
    h = hashlib.new(algo)
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

JUNK = {"desktop.ini", "Thumbs.db", ".DS_Store"}
paths = sorted(p for p in root.rglob("*") if p.is_file() and p.name != "MANIFEST.csv")
junk = [p for p in paths if p.name in JUNK or p.name.startswith(".") or ".ipynb_checkpoints" in p.parts]
assert not junk, junk
man = pd.DataFrame([dict(path=p.relative_to(root).as_posix(), bytes=p.stat().st_size,
                         md5=digest(p, "md5"), sha256=digest(p, "sha256")) for p in paths])
man.to_csv(root / "MANIFEST.csv", index=False)

dup = man[man.md5.duplicated(keep=False)]
assert dup.empty, f"Files with identical content (Dataverse flags duplicates): {dup.path.tolist()}"
top = man.assign(folder=man.path.str.rsplit("/", n=1).str[0].where(man.path.str.contains("/"), "."))
display(top.groupby("folder").agg(files=("path", "size"), MB=("bytes", lambda b: round(b.sum() / 1e6, 1))))
print(f"{len(man) + 1} files including MANIFEST.csv, {man.bytes.sum() / 1e6:,.1f} MB")
print("If the listing is right, set CONFIRM_UPLOAD = True in step 1, run step 1, then run step 7.")

In [ ]:
# 7 - Upload into the draft (re-runnable: files already in the draft with the same content are skipped)
assert CONFIRM_UPLOAD, "Check the listing in step 6, then set CONFIRM_UPLOAD = True in step 1"
import dvuploader as dv

def path_of(f):
    d = f.get("directoryLabel")
    return (d + "/" if d else "") + f["dataFile"]["filename"]

have = {path_of(f): f["dataFile"].get("md5") or f["dataFile"]["checksum"]["value"] for f in draft_files()}
want = dict(zip(man.path, man.md5)); want["MANIFEST.csv"] = digest(root / "MANIFEST.csv", "md5")
clash = [p for p in want if p in have and have[p] != want[p]]
assert not clash, f"Already in the draft with different content (delete them in the web interface first): {clash}"
todo = []
for p in sorted(p for p in want if p not in have):
    d = os.path.dirname(p)
    todo.append(dv.File(filepath=str(root / p), tab_ingest=False, **({"directory_label": d} if d else {})))
print(f"Uploading {len(todo)} files ({len(have)} already in the draft)")
if todo:
    dv.DVUploader(files=todo).upload(api_token=DV_TOKEN, dataverse_url=DV_URL,
                                     persistent_id=PID, n_parallel_uploads=2)

In [ ]:
# 8 - Check every file on the server against the manifest
srv = {}
for f in draft_files():
    c = f["dataFile"]["checksum"]
    srv[path_of(f)] = (c["type"], c["value"])
ALG = {"MD5": "md5", "SHA-1": "sha1", "SHA-256": "sha256", "SHA-512": "sha512"}
missing = sorted(set(want) - set(srv))
extra   = sorted(set(srv) - set(want))
differs = sorted(p for p in want if p in srv and digest(root / p, ALG[srv[p][0]]) != srv[p][1])
print(f"{len(srv)} files in draft, {len(want)} expected")
print("missing:", missing); print("extra:", extra); print("checksum differs:", differs)
assert not (missing or extra or differs), "Fix the files listed above before publishing"
print("All files match. Fill in the metadata and terms in the web interface, then click Publish.")